# 🔬 Notebook 3: Shopping Cart — Deep Dives (runnable)

## 🛠️ Setup

```bash
cd 06-system-designs/shopping-cart
uv sync
```

Select the `.venv` kernel in VS Code (top-right corner of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab only needs `pydantic` — no Redis, no database. We simulate everything in plain Python so you can focus on the *ideas*.


## What we'll build

Six mini-systems you can run and poke at:

1. **Concurrent updates from two devices** — the classic Dynamo lost-update, and
   three different ways to fix it. This is the heart of the lab.
2. **Inventory reservation with TTL** — why we don't decrement stock on *Add to Cart*.
3. **Checkout saga** — how to recover when payment fails after stock was reserved.
4. **Data hydration + short-TTL product cache** — how to show fresh prices without hammering the Product Service.
5. **Guest → user cart merge** — the idempotent algorithm, and the crash window that breaks a naive one.
6. **TTL & abandonment** — two clocks, and why native TTL is garbage collection rather than correctness.

We stay in pure Python — no Redis, no DB, no network. Every cell is standalone.

## Deep dive 1 — two devices, one cart

> This is the problem the Dynamo paper was written about. If you can only take
> one thing from this lab, take this section.

The setup is mundane. A shopper has your app open on their phone *and* your site
open on a laptop. Both loaded the cart a moment ago. They add something on each.

Nothing here is exotic — no partition, no crash, no clock skew. Just two writes
that overlap by a couple of hundred milliseconds.

In [ ]:
# ============ ❌ BAD: one row per cart, read-modify-write ================
class BlobCartDB:
    """PK = user_id, value = the whole {sku: qty} document. One row per cart."""
    def __init__(self):
        self.rows: dict[str, dict[str, int]] = {}
    def read(self, user):          return dict(self.rows.get(user, {}))
    def write(self, user, cart):   self.rows[user] = dict(cart)

db = BlobCartDB()
db.write("u1", {"book": 1})                 # yesterday's cart

phone  = db.read("u1")                      # 12:00:00.000 both devices load it
laptop = db.read("u1")                      # 12:00:00.010

phone["socks"] = 1                          # 12:00:01 phone: "add socks"
db.write("u1", phone)

laptop["lamp"] = 1                          # 12:00:02 laptop: "add lamp"
db.write("u1", laptop)                      #           …and overwrites the socks

print("final cart:", db.read("u1"))
assert db.read("u1") == {"book": 1, "lamp": 1}
print("❌ the socks are gone. Nothing failed, nothing was logged, and the")
print("   shopper only finds out at checkout — or after delivery.")

### Fix 1 — conditional write (optimistic concurrency)

Give each cart a version. A write says "…**and** the version is still 7". The
loser gets a rejection, re-reads, and retries. This is `UPDATE … WHERE version=?`,
DynamoDB's `ConditionExpression`, or an ETag/`If-Match` header.

In [ ]:
class VersionedCartDB:
    def __init__(self):
        self.rows: dict[str, tuple[dict, int]] = {}
    def read(self, user):
        cart, ver = self.rows.get(user, ({}, 0))
        return dict(cart), ver
    def write_if(self, user, cart, expected_version) -> bool:
        _, ver = self.rows.get(user, ({}, 0))
        if ver != expected_version:
            return False                     # someone else wrote first
        self.rows[user] = (dict(cart), ver + 1)
        return True

db = VersionedCartDB()
db.write_if("u1", {"book": 1}, 0)

phone,  v_phone  = db.read("u1")
laptop, v_laptop = db.read("u1")

phone["socks"] = 1
assert db.write_if("u1", phone, v_phone) is True        # phone wins the race

laptop["lamp"] = 1
assert db.write_if("u1", laptop, v_laptop) is False     # laptop is rejected…

laptop, v_laptop = db.read("u1")                        # …so it re-reads…
laptop["lamp"] = 1
assert db.write_if("u1", laptop, v_laptop) is True      # …and retries

print("final cart:", db.read("u1")[0])
assert db.read("u1")[0] == {"book": 1, "socks": 1, "lamp": 1}
print("✅ nothing lost.")
print()
print("What it costs:")
print("  • Every conflict is a full round trip and a retry. Under real contention")
print("    (one user hammering 'add' on a flaky connection) you get retry storms.")
print("  • It needs ONE authoritative replica to decide who won. That is a CP")
print("    choice — and in Notebook 1 we explicitly picked AP for the cart,")
print("    because rejecting an Add-to-Cart click is lost revenue.")
print("  • Across regions it is simply unavailable: you cannot do a conditional")
print("    write against a replica that cannot reach the leader.")

### Fix 2 — stop sending the whole cart. Send the *change*.

The lost update exists because the client shipped a full document that it had
computed from a stale read. If instead each item is its own row and the write is
`ADD qty :delta`, the database applies the change to whatever is currently there
— no read, nothing to be stale about.

This is exactly the `PK = user_id, SK = sku` key design from Notebook 2, plus
DynamoDB's atomic `ADD` (or Redis `HINCRBY`, or SQL `SET qty = qty + ?`).

In [ ]:
class ItemRowCartDB:
    """PK = user_id, SK = sku. One row per line item."""
    def __init__(self):
        self.rows: dict[tuple[str, str], int] = {}
    def add(self, user, sku, delta):
        """UPDATE cart SET qty = qty + :delta WHERE user=:u AND sku=:s — atomic."""
        self.rows[(user, sku)] = self.rows.get((user, sku), 0) + delta
    def remove(self, user, sku):
        self.rows.pop((user, sku), None)
    def get(self, user):
        return {s: q for (u, s), q in self.rows.items() if u == user and q > 0}

db = ItemRowCartDB()
db.add("u1", "book", 1)
# Both devices act on stale views — it no longer matters, they never read.
db.add("u1", "socks", 1)      # phone
db.add("u1", "lamp", 1)       # laptop
assert db.get("u1") == {"book": 1, "socks": 1, "lamp": 1}

# Even the same SKU from both devices commutes: the deltas simply sum.
db.add("u1", "book", 1)       # phone:  +1
db.add("u1", "book", 1)       # laptop: +1
assert db.get("u1")["book"] == 3
print("✅ concurrent adds:", db.get("u1"))

# ---- …but here is the case it does NOT fix -----------------------------
# The two devices are now on different replicas of a multi-region store, and
# the link between them is down. One removes an item; one increments it.
print()
print("Now partition the replicas. Phone removes 'book'; laptop adds one more.")
print("`remove` is a DELETE and `add` is an INCREMENT — they do not commute:")
for order in [("remove", "add"), ("add", "remove")]:
    d = ItemRowCartDB(); d.add("u1", "book", 1)
    for op in order:
        d.remove("u1", "book") if op == "remove" else d.add("u1", "book", 1)
    print(f"   replay order {order} → {d.get('u1')}")
print("❌ Two replicas that applied the same two operations in different orders")
print("   now disagree, forever. Atomic counters fixed concurrent *adds*; they")
print("   did nothing for concurrent add-vs-remove.")

### Fix 3 — what Dynamo actually did, and the bug it shipped with

Dynamo is AP: during a partition **both** replicas accept the write. When they
reconnect, neither version descends from the other — they are *concurrent* — so
Dynamo keeps both **siblings** and hands them to the application to reconcile.

Amazon's shopping cart reconciled by taking the **union** of the two item sets.
That choice is the reason for the anomaly the paper is honest about: *deleted
items sometimes come back*. Let's reproduce it rather than take their word.

In [ ]:
def descends(a: dict, b: dict) -> bool:
    """Does version vector `a` dominate `b`? (a saw everything b saw)"""
    return all(a.get(k, 0) >= v for k, v in b.items())

def concurrent(a: dict, b: dict) -> bool:
    return not descends(a, b) and not descends(b, a)

# Synced starting point: the cart holds one book, written by replica A.
base_items, base_vv = {"book": 1}, {"A": 1}

# --- partition ----------------------------------------------------------
# Phone talks to replica A and REMOVES the book.
a_items, a_vv = {}, {**base_vv, "A": 2}
# Laptop talks to replica B and ADDS a lamp (it never saw the removal).
b_items, b_vv = {**base_items, "lamp": 1}, {**base_vv, "B": 1}

assert concurrent(a_vv, b_vv), "neither replica saw the other's write"

# --- heal: reconcile the two siblings by union, exactly as Dynamo's cart did
reconciled = dict(b_items)                   # union of the two item sets
for sku, qty in a_items.items():
    reconciled[sku] = max(reconciled.get(sku, 0), qty)
print("sibling A (book removed):", a_items)
print("sibling B (lamp added)  :", b_items)
print("union-merged cart       :", reconciled)
assert reconciled["book"] == 1
print("❌ The book the shopper deleted is back in their cart.")
print("   Amazon shipped this on purpose: resurrecting a deleted item annoys a")
print("   customer, losing an added item loses a sale. Given a forced choice,")
print("   they took the annoyance. But it IS a forced choice only because the")
print("   data model cannot represent a deletion.")

### Fix 4 — make deletion representable: a CRDT cart

The union merge fails because a removed item is indistinguishable from an item
the other replica simply has not heard about yet. Fix the *model*, not the merge:
track, per SKU, **how many were added** and **how many were removed**, tagged by
which replica did it. That is a PN-Counter per SKU.

Merging is then element-wise `max`, which is commutative, associative and
idempotent — so replicas converge regardless of the order or number of times
updates arrive. No coordination, no rejected writes, still AP.

In [ ]:
class CRDTCart:
    """Per-SKU PN-Counter. `adds`/`rems` are keyed by (sku, replica_id)."""

    def __init__(self, replica: str):
        self.replica = replica
        self.adds: dict[tuple[str, str], int] = {}
        self.rems: dict[tuple[str, str], int] = {}

    def add(self, sku, n=1):
        k = (sku, self.replica); self.adds[k] = self.adds.get(k, 0) + n

    def remove(self, sku, n=1):
        # Never remove more than we can see, or a late-arriving add would be
        # cancelled by a removal that logically happened before it.
        n = min(n, self.qty(sku))
        k = (sku, self.replica); self.rems[k] = self.rems.get(k, 0) + n

    def qty(self, sku) -> int:
        a = sum(v for (s, _), v in self.adds.items() if s == sku)
        r = sum(v for (s, _), v in self.rems.items() if s == sku)
        return max(0, a - r)

    def items(self):
        skus = {s for s, _ in self.adds}
        return {s: self.qty(s) for s in sorted(skus) if self.qty(s) > 0}

    def merge(self, other: "CRDTCart") -> "CRDTCart":
        for k, v in other.adds.items(): self.adds[k] = max(self.adds.get(k, 0), v)
        for k, v in other.rems.items(): self.rems[k] = max(self.rems.get(k, 0), v)
        return self

    def clone(self, replica):
        c = CRDTCart(replica); c.adds = dict(self.adds); c.rems = dict(self.rems)
        return c

# --- same scenario as above ---------------------------------------------
synced = CRDTCart("A"); synced.add("book")          # the pre-partition state
phone, laptop = synced.clone("A"), synced.clone("B")

phone.remove("book")        # replica A: shopper deletes the book
laptop.add("lamp")          # replica B: shopper adds a lamp

merged = phone.clone("A").merge(laptop)
print("merged cart:", merged.items())
assert merged.items() == {"lamp": 1}, merged.items()
print("✅ the deletion survived the merge, and so did the lamp.")

# Convergence: merge order must not matter.
other_way = laptop.clone("B").merge(phone)
assert other_way.items() == merged.items()
# …and merging twice must change nothing (idempotent).
assert merged.clone("A").merge(laptop).items() == merged.items()
print("✅ commutative, associative, idempotent → replicas converge")

# Concurrent adds of the SAME sku are both preserved, unlike last-write-wins.
p, l = CRDTCart("A"), CRDTCart("B")
p.add("book"); l.add("book")
assert p.clone("A").merge(l).items() == {"book": 2}
print("✅ two devices each adding one book →", p.clone("A").merge(l).items())

### So which one do you actually ship?

| | Blob + read-modify-write | Conditional write | Per-item atomic `ADD` | CRDT cart |
|---|---|---|---|---|
| Lost update | **yes** ❌ | no | no | no |
| Concurrent add/remove | wrong | serialised | **wrong** ❌ | correct |
| Needs a leader | no | **yes** (CP) | single-partition only | no |
| Works multi-region AP | – | ✗ | ✗ | ✓ |
| Metadata cost | none | 1 int | none | grows with #replicas + tombstones |
| Complexity | trivial | low | low | **real** |

**The honest answer for most teams is the third column.** A cart lives in one
region, `PK=user_id, SK=sku` keeps every write on one partition, and atomic `ADD`
makes add/increment conflict-free. Add/remove races are rare enough, and their
consequence (an item lingers) mild enough, to accept.

Reach for the CRDT when the cart is genuinely multi-master — offline mobile that
must accept writes with no network, or active-active across regions. Then price
it properly: the metadata grows with every replica that ever touched the cart,
tombstones need garbage collecting behind a causal-stability watermark, and you
still **cannot** enforce "max 10 per item" globally, because a global invariant
needs coordination and coordination is the thing you gave up.

## Deep dive 2 — Inventory reservations with TTL

### The two naive choices (both bad)

| Strategy | Problem |
|---|---|
| Decrement stock on *Add to Cart* | Idle carts lock real buyers out for days |
| Decrement stock on *Pay* | Two people both pass the "in stock" check, one gets over-sold |

### The fix — reservation with TTL

At **checkout start** (not add-to-cart), create a **reservation** that decrements stock for a short window (e.g. 10 minutes). If payment succeeds, convert reservation → permanent sale. If TTL expires, the reservation **auto-releases** and stock goes back.

The class below simulates the inventory service. Notice the explicit `release()` method — we don't reach into its internals from outside.

In [ ]:
import time, threading, uuid

class Inventory:
    """In-memory inventory with time-boxed reservations."""
    def __init__(self, stock: dict[str, int]):
        self.stock = dict(stock)              # sku -> available units
        self.reservations: dict[str, tuple[str, int, float]] = {}
        self.lock = threading.Lock()          # single-machine concurrency

    def reserve(self, rid: str, sku: str, qty: int, ttl: float = 600) -> bool:
        with self.lock:
            self._sweep()
            if self.stock.get(sku, 0) < qty:
                return False
            self.stock[sku] -= qty
            self.reservations[rid] = (sku, qty, time.time() + ttl)
            return True

    def confirm(self, rid: str) -> bool:
        """Payment succeeded — reservation becomes a permanent sale."""
        with self.lock:
            return self.reservations.pop(rid, None) is not None

    def release(self, rid: str) -> bool:
        """Explicit rollback (payment failed / user cancelled)."""
        with self.lock:
            entry = self.reservations.pop(rid, None)
            if entry is None:
                return False
            sku, qty, _ = entry
            self.stock[sku] += qty
            return True

    def _sweep(self) -> None:
        """Reclaim stock from expired reservations."""
        now = time.time()
        for rid in [k for k, (_, _, exp) in self.reservations.items() if exp < now]:
            sku, qty, _ = self.reservations.pop(rid)
            self.stock[sku] += qty

# Scenario: 2 units in stock, Alice reserves 1 with a 1 s TTL and ghosts.
inv = Inventory({"BOOK": 2})
print("alice reserves 1:", inv.reserve("res-alice", "BOOK", 1, ttl=1))
print("bob tries 2     :", inv.reserve("res-bob1",  "BOOK", 2))   # only 1 left
print("bob takes 1     :", inv.reserve("res-bob2",  "BOOK", 1))
print("stock now       :", inv.stock)
print()
print("... 1.1 s passes, alice's TTL expires ...")
time.sleep(1.1)
inv._sweep()
print("stock now       :", inv.stock)   # alice's unit came back

## Deep dive 3 — Checkout saga with clean compensation

Checkout spans **Inventory → Payment → Order**. You can't wrap them in one DB transaction (they're separate services). Instead: run them in sequence, and if one fails, **compensate** the previous step.

```
┌─ reserve stock ─► charge card ─► create order ─┐
│       │                │                       │
│       │(fail)          │(fail)                 ▼
▼       ▼                ▼                    success
abort  ✓           release() stock
```

Below we use the **clean `release()` API** instead of poking internals. This makes the saga readable and testable.

In [ ]:
class FakeCard:
    """Payment gateway stub."""
    def __init__(self, ok: bool = True): self.ok = ok
    def charge(self, amount: int) -> dict:
        if not self.ok:
            raise RuntimeError("card declined")
        return {"txn": f"tx_{amount}"}

def checkout(inv: Inventory, card: FakeCard,
             cart_id: str, sku: str, qty: int, amount: int) -> dict:
    rid = f"res-{cart_id}"

    # Step 1: reserve stock
    if not inv.reserve(rid, sku, qty, ttl=60):
        return {"status": "out_of_stock"}

    # Step 2: charge card (with compensation on failure)
    try:
        txn = card.charge(amount)
    except Exception as e:
        inv.release(rid)                           # ← compensate
        return {"status": "payment_failed", "error": str(e)}

    # Step 3: confirm the reservation (permanent decrement)
    inv.confirm(rid)
    # (Real systems would also write an Order record here.)
    return {"status": "paid", "txn": txn, "cart_id": cart_id}

inv = Inventory({"BOOK": 5})
print(checkout(inv, FakeCard(ok=True),  "c1", "BOOK", 2, 20))
print(checkout(inv, FakeCard(ok=False), "c2", "BOOK", 1, 10))   # fails, releases
print(checkout(inv, FakeCard(ok=True),  "c3", "BOOK", 1, 10))
print("final stock:", inv.stock)   # 5 - 2 - 1 = 2

### Why not 2PC (two-phase commit)?

2PC *can* coordinate a transaction across services, but it blocks resources while voting and falls apart when any coordinator dies. Sagas trade strict atomicity for **availability and partial-failure recovery**, which is exactly what e-commerce wants.

## Deep dive 4 — Data hydration + short-TTL product cache

Remember from NB1: the Cart DB stores `{sku, qty}` only. At **view** time, the Cart Service must fetch the current price/name from the **Product Service**. That's the **hydration** step.

### Why you need a cache in front

If every cart view calls the Product Service for every item, and carts are viewed 150 000× / second at peak, the Product Service melts. Put a **5-minute TTL** in front of it. Prices only need to be accurate *to the minute*; the hard price check happens at checkout anyway.

In [ ]:
import time
from decimal import Decimal

# --- Product Service stub -------------------------------------------------
class ProductService:
    def __init__(self, catalog: dict[str, dict]):
        self.catalog = catalog
        self.calls = 0                           # count network hits

    def get(self, sku: str) -> dict:
        self.calls += 1                          # simulate a network call
        return self.catalog[sku].copy()

# --- Short-TTL cache ------------------------------------------------------
class ProductCache:
    def __init__(self, svc: ProductService, ttl: float = 300):
        self.svc = svc
        self.ttl = ttl
        self.cache: dict[str, tuple[dict, float]] = {}

    def get(self, sku: str) -> dict:
        now = time.time()
        entry = self.cache.get(sku)
        if entry and now - entry[1] < self.ttl:
            return entry[0]                      # cache hit
        fresh = self.svc.get(sku)                # miss → fetch + store
        self.cache[sku] = (fresh, now)
        return fresh

# --- Cart (slim) + hydration ---------------------------------------------
def view_cart(cart_items: list[dict], cache: ProductCache) -> dict:
    hydrated, total = [], Decimal(0)
    for it in cart_items:
        p = cache.get(it["sku"])
        line_total = Decimal(str(p["price"])) * it["qty"]
        hydrated.append({**it, "name": p["name"], "price": p["price"],
                         "line_total": str(line_total)})
        total += line_total
    return {"items": hydrated, "total": str(total)}

svc = ProductService({
    "A": {"name": "Python Crash Course", "price": "29.99"},
    "B": {"name": "Clean Code",          "price": "35.00"},
})
cache = ProductCache(svc, ttl=300)

slim_cart = [{"sku": "A", "qty": 2}, {"sku": "B", "qty": 1}]

print("first view  :", view_cart(slim_cart, cache))
print("product-svc calls so far:", svc.calls)   # 2

print("second view :", view_cart(slim_cart, cache))
print("product-svc calls so far:", svc.calls)   # still 2 — cache hit

### Graceful degradation

What if the Product Service is **down** when a user opens their cart?

- **Bad:** 500 error, user sees nothing.
- **Best:** show cached data if we have it (even if stale); if no cache, show items with `"Loading price..."` placeholders and a banner. The cart itself (sku + qty) still works because it's in Redis.

This is called **graceful degradation** — degrade the feature, don't take the page down. Try implementing the stale-on-error path as an exercise.

## Deep dive 5 — Guest → User cart merge

Scenario: a shopper browses anonymously (`guest_session_id = "gs_abc"`), adds items, then logs in.

The reference spec:
- **Overlap** (same sku in both carts) → **sum quantities**, capped at the per-item limit.
- **No overlap** → add guest's items to the user cart.
- **Delete the guest cart** after merge so a retry doesn't double up.
- The whole thing must be **idempotent** — the client may retry on a flaky connection.

In [ ]:
from dataclasses import dataclass, field

MAX_QTY_PER_ITEM = 10
MAX_ITEMS        = 50          # same caps as the Pydantic model in Notebook 2

@dataclass
class CartStore:
    """Toy cart DB. Keyed by user_id *or* guest_session_id — both strings."""
    carts: dict[str, dict[str, int]] = field(default_factory=dict)  # owner -> {sku:qty}
    merges: dict[str, dict] = field(default_factory=dict)           # merge_id -> result

    def get(self, owner: str) -> dict[str, int]:
        return self.carts.get(owner, {}).copy()

    def put(self, owner: str, items: dict[str, int]) -> None:
        self.carts[owner] = dict(items)          # store a COPY, never an alias

    def delete(self, owner: str) -> None:
        self.carts.pop(owner, None)

    def _transact(self, user_id, merged, merge_id) -> None:
        """One atomic write covering BOTH the new cart and the merge marker.
        DynamoDB: TransactWriteItems. SQL: one transaction. Redis: MULTI/EXEC.
        Splitting these two writes is the bug demonstrated in the next cell."""
        self.carts[user_id] = dict(merged)
        self.merges[merge_id] = dict(merged)

    def compute_merge(self, guest_id, user_id):
        merged = self.get(user_id)
        for sku, qty in self.get(guest_id).items():
            merged[sku] = min(merged.get(sku, 0) + qty, MAX_QTY_PER_ITEM)
        if len(merged) > MAX_ITEMS:
            # Keep the user's own items first — dropping what they saved last
            # week to make room for a guest session would be indefensible.
            # In production this returns a warning to the client, not silence.
            merged = dict(list(merged.items())[:MAX_ITEMS])
        return merged

    def merge(self, guest_id: str, user_id: str, merge_id: str) -> dict[str, int]:
        # 🔑 Idempotency: same merge_id returns the same answer.
        if merge_id in self.merges:
            self.delete(guest_id)        # the crash may have been the cleanup itself
            return dict(self.merges[merge_id])

        merged = self.compute_merge(guest_id, user_id)
        self._transact(user_id, merged, merge_id)   # ← atomic: cart + marker
        self.delete(guest_id)                       # ← best-effort, safe to repeat
        return dict(merged)

# Scenario: guest has 2×A, 1×B. User (from last week) has 1×A, 1×C.
db = CartStore()
db.put("gs_abc", {"A": 2, "B": 1})
db.put("u_42",   {"A": 1, "C": 1})

first = db.merge("gs_abc", "u_42", "m-001")
print("1st merge :", first)              # {'A': 3, 'C': 1, 'B': 1}
print("retry     :", db.merge("gs_abc", "u_42", "m-001"))
assert db.merge("gs_abc", "u_42", "m-001") == first

# The idempotency record must not alias the live cart, or a later add corrupts
# the replay answer for every subsequent retry.
db.put("u_42", {"A": 9})
assert db.merges["m-001"] == {"A": 3, "C": 1, "B": 1}, db.merges["m-001"]
print("✅ replay record survived a later write to the cart")

# Hoarding attempt: guest with 50 copies of A
db.put("gs_hoard", {"A": 50})
db.put("u_99", {"A": 2})
print("cap       :", db.merge("gs_hoard", "u_99", "m-002"))   # {'A': 10}

### The crash window a naive merge leaves open

Look closely at the order of writes. A merge does three things: write the user's
new cart, delete the guest cart, record the merge marker. If the marker is
written **last** and the process dies before it, the retry finds no marker and
merges the guest cart in **again** — quantities double.

The fix is not "retry harder", it is to make the cart write and the marker write
**one atomic operation**, and to make the guest-cart delete the only thing that
can be left dangling — because deleting twice is harmless.

In [ ]:
class Crash(RuntimeError): pass

def merge_naive(db, guest_id, user_id, merge_id, crash_after_cart=False):
    """The tempting ordering: cart → delete guest → marker. Three writes."""
    if merge_id in db.merges:
        return dict(db.merges[merge_id])
    merged = db.compute_merge(guest_id, user_id)
    db.put(user_id, merged)
    if crash_after_cart:
        raise Crash("pod evicted between the cart write and the marker write")
    db.delete(guest_id)
    db.merges[merge_id] = dict(merged)
    return merged

# --- ❌ naive ordering + one crash --------------------------------------
db = CartStore(); db.put("gs_x", {"A": 2}); db.put("u_1", {"A": 1})
try:
    merge_naive(db, "gs_x", "u_1", "m-9", crash_after_cart=True)
except Crash as e:
    print("crash:", e)
print("client retries with the same merge_id…")
print("  result:", merge_naive(db, "gs_x", "u_1", "m-9"))
assert db.get("u_1") == {"A": 5}, db.get("u_1")
print("❌ the guest's 2 were applied twice: 1 + 2 + 2 = 5, not 3.")

# --- ✅ atomic cart+marker, then best-effort cleanup ---------------------
db = CartStore(); db.put("gs_x", {"A": 2}); db.put("u_1", {"A": 1})
merged = db.compute_merge("gs_x", "u_1")
db._transact("u_1", merged, "m-9")        # atomic; if we die before this, nothing happened
print("\ncrash: pod evicted right after the transaction, before the cleanup")
print("client retries with the same merge_id…")
print("  result:", db.merge("gs_x", "u_1", "m-9"))
assert db.get("u_1") == {"A": 3}, db.get("u_1")
assert "gs_x" not in db.carts, "the retry finished the cleanup"
print("✅ 1 + 2 = 3, and the orphaned guest cart got swept up by the retry.")

# --- and if the client generates a NEW merge_id on retry? ---------------
# Common, and the design survives it: the guest cart is already gone, so the
# second merge is a no-op rather than a double-add. Self-healing by construction.
print("\nretry with a fresh merge_id:", db.merge("gs_x", "u_1", "m-10"))
assert db.get("u_1") == {"A": 3}
print("✅ a regenerated merge_id is harmless — the guest cart is the real token.")

## Deep dive 6 — TTL and abandonment: two clocks, not one

Carts expire. That sounds like one setting; it is actually two independent
clocks that people constantly conflate:

- the **marketing clock** — "untouched for 1 hour" → send an abandoned-cart email;
- the **storage clock** — "untouched for 30 days" → delete the row.

And a third thing that is neither: **native TTL is garbage collection, not
correctness.** DynamoDB documents deletion "typically within 48 hours" of the
expiry timestamp. If your code trusts the absence of a row to mean "expired",
you will serve month-old carts for two days after they should have died.

In [ ]:
from dataclasses import dataclass, field

HOUR, DAY = 3600, 86_400
ABANDON_AFTER = 1 * HOUR      # marketing clock
EXPIRE_AFTER  = 30 * DAY      # storage clock

@dataclass
class CartRow:
    user: str
    items: dict
    updated_at: float
    expires_at: float

class TtlCartStore:
    def __init__(self):
        self.rows: dict[str, CartRow] = {}
        self.sweeps = 0

    def write(self, user, items, now, refresh_ttl=True):
        row = self.rows.get(user)
        expires = (now + EXPIRE_AFTER) if (refresh_ttl or row is None) else row.expires_at
        self.rows[user] = CartRow(user, dict(items), now, expires)

    def get(self, user, now):
        """Read-time expiry filter. Do NOT rely on the sweeper having run."""
        row = self.rows.get(user)
        if row is None or row.expires_at <= now:
            return None
        return row

    def sweep(self, now):
        """What the store's background TTL process does — eventually."""
        self.sweeps += 1
        dead = [u for u, r in self.rows.items() if r.expires_at <= now]
        for u in dead:
            del self.rows[u]
        return len(dead)

# ---- ❌ the bug: TTL stamped once, at cart creation ---------------------
s = TtlCartStore()
t0 = 0.0
s.write("shopper", {"A": 1}, now=t0)
for day in range(1, 40):                                   # they keep shopping…
    s.write("shopper", {"A": day}, now=t0 + day * DAY, refresh_ttl=False)
now = t0 + 40 * DAY
assert s.get("shopper", now) is None
print("❌ TTL set only on creation: an actively-used cart expired on day 30")
print("   while the shopper was mid-session.")

# ---- ✅ every write pushes the TTL forward -----------------------------
s = TtlCartStore()
s.write("shopper", {"A": 1}, now=t0)
for day in range(1, 40):
    s.write("shopper", {"A": day}, now=t0 + day * DAY)     # refresh_ttl=True
assert s.get("shopper", t0 + 40 * DAY) is not None
print("✅ TTL refreshed on write: still alive on day 40")

# ---- native TTL is eventual: the read-time filter is what protects you --
s = TtlCartStore()
s.write("ghost", {"A": 1}, now=t0)
later = t0 + EXPIRE_AFTER + 2 * DAY        # 2 days past expiry, sweeper hasn't run
assert "ghost" in s.rows and s.sweeps == 0        # the row is physically there…
assert s.get("ghost", later) is None              # …but the read filter hides it
assert s.sweep(later) == 1                        # the sweeper catches up later
print("✅ expired-but-not-yet-deleted rows are invisible to reads")

In [ ]:
# ---- The abandonment funnel: classify, don't just delete ---------------
import random
random.seed(3)

s = TtlCartStore()
NOW = 100 * DAY
population = []
for i in range(20_000):
    # realistic-ish: most carts were touched recently, a long tail was not
    idle = random.choice([random.uniform(0, HOUR)] * 5 + [random.uniform(HOUR, 30 * DAY)] * 4
                         + [random.uniform(30 * DAY, 90 * DAY)])
    s.write(f"u{i}", {"A": 1}, now=NOW - idle)
    population.append(idle)

def classify(store, now):
    live = abandoned = expired = 0
    for row in store.rows.values():
        idle = now - row.updated_at
        if row.expires_at <= now:  expired += 1
        elif idle >= ABANDON_AFTER: abandoned += 1
        else:                       live += 1
    return live, abandoned, expired

live, abandoned, expired = classify(s, NOW)
total = len(s.rows)
print(f"{'active (touched < 1h)':<32}{live:>8,}  {live/total:>6.1%}")
print(f"{'abandoned → email campaign':<32}{abandoned:>8,}  {abandoned/total:>6.1%}")
print(f"{'expired → reclaim the row':<32}{expired:>8,}  {expired/total:>6.1%}")
print()
first_pass  = s.sweep(NOW)
second_pass = s.sweep(NOW)
print(f"sweeper reclaimed {first_pass:,} rows; a second pass reclaimed "
      f"{second_pass} — sweeping is idempotent, so it can run as often as you like")
print()
print("Two clocks, two owners: the 1-hour bucket belongs to marketing and must")
print("NOT delete anything; the 30-day bucket belongs to storage and must not")
print("send anything. Wiring them to the same threshold is how you end up")
print("emailing 'you left something in your cart!' about a cart you just deleted.")

### The parts that bite in production

- **Resurrection.** A stale replica or a retried write can re-create a cart that
  TTL already reclaimed. Keep `expires_at` in the row and filter at read time
  (as above); an old write with an old `expires_at` is then harmless.
- **Deleting the cart is not deleting the reservation.** If checkout started, an
  inventory reservation exists with its own, much shorter TTL. Those two
  lifecycles are independent — deep dive 2 owns the second one.
- **TTL on a cache is not TTL on the source of truth.** The Redis entry
  disappearing after 24 h of inactivity should cause a re-read from DynamoDB,
  not an empty cart. A cache miss must never be reported to the user as "empty".
- **Guest carts are the real volume.** Most guest sessions never convert, so
  guest carts dominate row count and deserve the aggressive (7-day) TTL, while
  logged-in carts get 30+ days.

## Closing thoughts

| Pattern | Where |
|---|---|
| **Send the delta, not the document** | Any concurrently-edited record (carts, counters, likes, inventory) |
| **CRDT / commutative merge** | Anything multi-master: offline mobile, active-active regions |
| **Slim source-of-truth + live hydration** | Any service where a field can change (prices, permissions, user names) |
| **Reservation with TTL** | Any shared scarce resource (event seats, ride drivers, ad impressions) |
| **Saga with compensation** | Any cross-service workflow that can't share a DB transaction |
| **Idempotency keys + atomic marker write** | Any retryable action that moves money or mutates state |
| **Short-TTL metadata cache** | Any fan-out that amplifies backend load |
| **Per-item qty caps** | Any user-controlled collection (carts, follows, uploads) |

### The one-sentence version

A cart is a **conflict-resolution problem wearing an e-commerce costume**: pick
`PK=user_id, SK=sku` with atomic `ADD` so the common concurrent case cannot lose
a write, be explicit that add-vs-remove races are the residual risk you accepted,
and reach for a CRDT only when the cart is genuinely multi-master.

### Exercises

1. Extend `CRDTCart` with garbage collection: once every replica has seen an
   entry, the `(sku, replica)` pair can be compacted. What do you need to track?
2. Add a **stale-on-error** fallback to `ProductCache` — if the upstream call
   fails, serve the last cached value with a `stale=True` flag.
3. Extend `Inventory` so `reserve` is atomic across processes (hint: a Redis Lua
   script, or `WATCH/MULTI/EXEC`).
4. Write a test that simulates 1000 concurrent checkouts of the last unit in
   stock — only one should win.
5. Make `merge` work when the guest cart is itself a `CRDTCart` from an offline
   mobile session. What is the right merge_id then?